In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Get a summary of the dataset
print(train_data.info())

# Check for missing values
print(train_data.isnull().sum())

# Describe the dataset
print(train_data.describe())

# Distinguish column types
numeric_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object']).columns

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

# Visualize numeric columns
for col in numeric_cols:
    plt.figure(figsize=(10, 6))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize categorical columns
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(y=train_data[col])
    plt.title(f'Count of {col}')
    plt.xlabel('Count')
    plt.ylabel(col)
    plt.show()

# Correlation matrix for numeric columns
plt.figure(figsize=(12, 8))
corr_matrix = train_data[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


       id  store_sales  unit_sales  ...  prepared_food  florist    cost
0  177400         8.49           3  ...              0        0  133.42
1   35064         4.47           3  ...              0        0  109.06
2  268168         6.84           3  ...              1        1   91.58
3  177181         4.54           2  ...              0        0  124.36
4  271461         8.34           3  ...              1        1   96.55

[5 rows x 17 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28827 entries, 0 to 28826
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    28827 non-null  int64  
 1   store_sales           28827 non-null  float64
 2   unit_sales            28827 non-null  int64  
 3   total_children        28827 non-null  int64  
 4   num_children_at_home  28827 non-null  int64  
 5   avg_cars_at_home      28827 non-null  int64  
 6   gross_weight          2

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:47:00.272 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'store_sales', 'unit_sales', 'total_children', 'num_children_at_home', 'avg_cars_at_home', 'gross_weight', 'recyclable_package', 'low_fat', 'units_per_case', 'store_sqft', 'coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist', 'cost'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale, LabelEncode

# Load the test data
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/media_campaign_cost/test.csv')

# Copy the dataframes to avoid modifying the original data
train_data_copy = train_data.copy()
test_data_copy = test_data.copy()

# Handle missing values
numeric_cols = train_data_copy.select_dtypes(include=[np.number]).columns
categorical_cols = train_data_copy.select_dtypes(include=['object']).columns

# Fill missing values for numeric columns with mean
fill_missing_numeric = FillMissingValue(features=numeric_cols, strategy='mean')
train_data_copy = fill_missing_numeric.fit_transform(train_data_copy)
test_data_copy = fill_missing_numeric.transform(test_data_copy)

# Fill missing values for categorical columns with most frequent value
fill_missing_categorical = FillMissingValue(features=categorical_cols, strategy='most_frequent')
train_data_copy = fill_missing_categorical.fit_transform(train_data_copy)
test_data_copy = fill_missing_categorical.transform(test_data_copy)

# Encode categorical variables using label encoding
label_encode = LabelEncode(features=categorical_cols)
train_data_copy = label_encode.fit_transform(train_data_copy)
test_data_copy = label_encode.transform(test_data_copy)

# Normalize numerical features
standard_scale = StandardScale(features=numeric_cols)
train_data_copy = standard_scale.fit_transform(train_data_copy)
test_data_copy = standard_scale.transform(test_data_copy)

# Display the first few rows of the processed data
print(train_data_copy.head())
print(test_data_copy.head())


         id  store_sales  unit_sales  ...  prepared_food   florist      cost
0 -0.018151     0.649551   -0.052892  ...      -1.003579 -1.006788  1.130897
1 -1.392321    -0.568748   -0.052892  ...      -1.003579 -1.006788  0.319468
2  0.858160     0.149503   -0.052892  ...       0.996433  0.993258 -0.262790
3 -0.020265    -0.547533   -1.325621  ...      -1.003579 -1.006788  0.829109
4  0.889952     0.604092   -0.052892  ...       0.996433  0.993258 -0.097240

[5 rows x 17 columns]
         id  store_sales  unit_sales  ...  prepared_food   florist      cost
0 -0.054761    -1.165775   -1.325621  ...      -1.003579  0.993258 -0.800412
1  0.993766    -0.741491   -0.052892  ...       0.996433  0.993258  1.075270
2 -1.670995    -0.832409   -0.052892  ...      -1.003579 -1.006788 -1.341698
3  0.641486     2.489122    1.219837  ...      -1.003579 -1.006788 -0.434669
4 -1.386480    -1.396100   -0.052892  ...      -1.003579 -1.006788  0.804793

[5 rows x 17 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'store_sales', 'unit_sales', 'total_children', 'num_children_at_home', 'avg_cars_at_home', 'gross_weight', 'recyclable_package', 'low_fat', 'units_per_case', 'store_sqft', 'coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist', 'cost'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from xgboost import XGBRegressor

# Assuming train_data_copy and test_data_copy are already preprocessed
X = train_data_copy.drop(columns=['cost'])
y = train_data_copy['cost']

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost model
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=100)

# Predict on the validation set
y_val_pred = model.predict(X_val)

# Calculate RMSE
rmse = np.sqrt(mean_squared_log_error(y_val, y_val_pred))
print(f'Validation RMSE: {rmse}')

# Predict on the test set
X_test = test_data_copy.drop(columns=['cost'])
y_test_pred = model.predict(X_test)

# Save the predictions to a CSV file
submission = pd.DataFrame({'id': test_data['id'], 'cost': y_test_pred})
submission.to_csv('submission.csv', index=False)


TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'